In [ ]:
%matplotlib qt
import load_params
import optim_model
from matplotlib import pyplot as plt
from time import perf_counter

# import post_processing
import os
from tabulate import tabulate
import numpy as np
import json
import borefield_params
import helper_functions as hf


# Run the size optimization
result_dict, load_params_time, size_optim_time = hf.run_size_optim()


In [ ]:
# Convert all numpy arrays in the result dict to lists:
def convert_numpy_to_list(obj):
    """
    Recursively converts all numpy arrays in a dictionary (or list) to lists.
    
    Args:
        obj (dict, list, or other): The object to process.
        
    Returns:
        The processed object with numpy arrays converted to lists.
    """
    if isinstance(obj, dict):  # If the object is a dictionary
        return {key: convert_numpy_to_list(value) for key, value in obj.items()}
    elif isinstance(obj, list):  # If the object is a list
        return [convert_numpy_to_list(item) for item in obj]
    elif isinstance(obj, np.ndarray):  # If the object is a numpy array
        return obj.tolist()
    else:  # Return the object as-is for all other types
        return obj
    
result_dict = convert_numpy_to_list(result_dict)

# Save results as a JSON file
with open("result_dict.json", "w") as file:
    json.dump(result_dict, file, indent=4)

In [ ]:
# Post processing
# General
print("---------- General Results ----------")
table = [
    ["tac", result_dict["tac"],  "euro"],
    ["co2", result_dict["co2"], "tonCO2/year"],
    ["CAPEX initial", result_dict["CAPEX_initial"], "euro"],
    ["CAPEX annualized", result_dict["CAPEX_annual"], "euro/year"],
    ["OPEX yearly total", result_dict["OPEX_yearly"], "euro/year"],
    ["OPEX yearly energy and CO2", result_dict["OPEX_yearly_energy"], "euro/year"],
    ["OPEX yearly maintenance", result_dict["OPEX_yearly_maint"], "euro/year"],
    ["Electricity offtake", result_dict["from_el_grid_total"], "MWh/year"],
    ["Electricity injection", result_dict["to_el_grid_total"], "MWh/year"],
]

print(tabulate(table, headers=["KPI", "Value", "Unit"], tablefmt="grid"))

# Devices

for dev in result_dict["installed_devices"]:
    print("---------- Results", dev, "----------")
    
    for key in result_dict[dev]:
        print(key, ": ",result_dict[dev][key])

In [ ]:
print(result_dict["GSHP"].keys())
QCon_GSHP_sizeopt = result_dict["GSHP"]["heat"]
QCon_ASHP_sizeopt = result_dict["ASHP"]["heat"]
path_input_data = r"C:\Workdir\Develop\IOCS\input_data"
COP_GSHP_sizeopt = np.loadtxt(os.path.join(path_input_data, "COP_GSHP.txt"))
COP_ASHP_sizeopt = np.loadtxt(os.path.join(path_input_data, "COP_ASHP.txt"))


path_outputsAll = r"C:\Users\u0148284\OneDrive - KU Leuven\Taco\ResultFiles\IOCS\StijnStreuvel_wComf10000_TMY\Iteration5\outputsAll.csv"
path_OutputNames = r"C:\Users\u0148284\OneDrive - KU Leuven\Taco\ResultFiles\IOCS\StijnStreuvel_wComf10000_TMY\Iteration5\OutputNames.txt"
df_TACO = hf.read_ocp_result(path_outputsAll, path_OutputNames) 
QCon_GSHP_TACO = df_TACO["QCon_GSHP"].to_numpy()/1e3
QCon_ASHP_TACO = df_TACO["QCon_ASHP"].to_numpy()/1e3
COP_GSHP_TACO = df_TACO["COP_GSHP"].to_numpy()
COP_ASHP_TACO = df_TACO["COP_ASHP"].to_numpy()
TSup = df_TACO["T_COP_Sup"].to_numpy()
Tair = df_TACO["T_COP_Air"].to_numpy()
TBor = df_TACO["T_COP_Bor"].to_numpy()


# Calculate daily sums
QCon_ASHP_sizeopt_daily = np.sum(np.array(QCon_ASHP_sizeopt).reshape(-1, 24), axis=1)
QCon_ASHP_TACO_daily = np.sum(QCon_ASHP_TACO.reshape(-1, 24), axis=1)
QCon_GSHP_sizeopt_daily = np.sum(np.array(QCon_GSHP_sizeopt).reshape(-1, 24), axis=1)
QCon_GSHP_TACO_daily = np.sum(QCon_GSHP_TACO.reshape(-1, 24), axis=1)

plt.figure("Compare sizeopt vs TACO")
nb_subplots = 3
ax1 = plt.subplot(nb_subplots, 1, 1)
plt.plot(QCon_GSHP_sizeopt, label="GSHP Sizeopt", color="blue")
plt.plot(QCon_GSHP_TACO, label="GSHP Taco", color="blue", linestyle="--")
plt.plot(QCon_ASHP_sizeopt, label="ASHP Sizeopt", color="orange")
plt.plot(QCon_ASHP_TACO, label="ASHP Taco", color="orange", linestyle="--")
plt.ylabel("QCon [kW]"), plt.title("heating power"), plt.legend(), plt.grid()
ax2 = plt.subplot(nb_subplots, 1, 2, sharex=ax1)
plt.plot(COP_GSHP_sizeopt, label="GSHP Sizeopt", color="blue")
plt.plot(COP_GSHP_TACO, label="GSHP Taco", color="blue", linestyle="--")
plt.plot(COP_ASHP_sizeopt, label="ASHP Sizeopt", color="orange")
plt.plot(COP_ASHP_TACO, label="ASHP Taco", color="orange", linestyle="--")
plt.ylabel("COP"), plt.title("COP"), plt.legend(), plt.grid()
ax3 = plt.subplot(nb_subplots, 1, 3, sharex=ax1)
plt.plot(TSup, label="Taco Sup", color="blue")
plt.plot(Tair, label="Taco Air", color="orange")
plt.plot(TBor, label="Taco Bore", color="green")
plt.ylabel("T [°C]"), plt.title("Temperatures"), plt.legend(), plt.grid()

plt.show()

In [ ]:
%matplotlib qt
print("result_dict", result_dict["Borefield"].keys())

plt.figure()
plt.plot(result_dict["GSHP"]["heat"], label="GSHP")
plt.legend(), plt.grid()

plt.figure()
plt.plot(result_dict["Borefield"]["Qb_extraction"], label="Qb_extraction")
plt.plot(result_dict["Borefield"]["Qb_peak_extraction"], label="Qb_peak_extraction")
plt.legend(), plt.grid()


plt.figure()
plt.plot(result_dict["Borefield"]["Tf_monthly_extraction"], label="Tf_monthly_extraction")
plt.plot(result_dict["Borefield"]["Tf_peak_extraction"], label="Tf_peak_extraction")
plt.legend(), plt.grid()
plt.show()

print(result_dict["Borefield"]["cap"])

# # export heat borefield to csv
# Qb_opt = np.tile(result_dict["Borefield"]["heat"]*1000, 20)
# data_to_export = np.column_stack((time_in_seconds, Qb_opt))
# output_csv_path = "QBor_solution.csv"
# np.savetxt(output_csv_path, data_to_export, delimiter=";", header=f"#1\ndouble data({time_steps}, 2)", comments='', fmt=['%d', '%.1f'])

# export fluid temperature to csv



In [ ]:
import GHEtool as ghe
import numpy as np

QBor = -np.array(result_dict["Borefield"]["heat"])*1000 + np.array(result_dict["Borefield"]["cool"])*1000 + np.array(result_dict["Borefield"]["reg"])*1000
borefield = ghe.Borefield()

# set ground data in borefield
borefield.set_ground_parameters(devs["Borefield"]["bor_params"]["ground_data"])
borefield.set_fluid_parameters(devs["Borefield"]["bor_params"]["fluid_data"])
borefield.set_pipe_parameters(devs["Borefield"]["bor_params"]["pipe_data"])

borefield.create_rectangular_borefield(N_1=devs["Borefield"]["bor_params"]["N_1"], N_2=devs["Borefield"]["bor_params"]["N_2"], B_1=devs["Borefield"]["bor_params"]["B"], B_2=devs["Borefield"]["bor_params"]["B"], \
                                            H=100, D=devs["Borefield"]["bor_params"]["D"], r_b=devs["Borefield"]["bor_params"]["r_b"])

# set temperature bounds
borefield.set_max_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_max"])
borefield.set_min_avg_fluid_temperature(devs["Borefield"]["bor_params"]["Tf_min"])

# load the hourly profile
borefield.simulation_period = devs["Borefield"]["bor_params"]["lifetime"]
load = ghe.HourlyGeothermalLoad(simulation_period=devs["Borefield"]["bor_params"]["lifetime"])
hea_load = np.where(QBor <= 0, -QBor, 0)
coo_load = np.where(QBor > 0, QBor, 0)
load.hourly_extraction_load = hea_load / 1000  # convert to kW
load.hourly_injection_load = coo_load / 1000  # convert to kW
borefield.load = load
load.all_months_equal = True

colors = plt.cm.tab10(np.linspace(0, 1, 10))
Tb_GHE_monthly = {}
Tf_GHE_monthly = {}
Tb_GHE_hourly_ST = {}
Tf_GHE_hourly_ST = {}

## options without short-term effects
options = {
            "nSegments": 12,
            "segment_ratios": None,
            "disp": False,
            "profiles": True,
            "method": "equivalent",
            "cylindrical_correction": True,
        }
## set options
borefield.set_options_gfunction_calculation(options)



borefield._calculate_temperature_profile(H=result_dict["Borefield"]["length"], hourly=False)
    
Tb_GHE_monthly = np.repeat(borefield.results.Tb, 730)
Tf_GHE_monthly = np.repeat(borefield.results.peak_extraction, 730)

plt.figure("Compare GHE and optimization")
nb_plots = 2
plt.subplot(nb_plots, 1, 1)
plt.plot(Tb_GHE_monthly, label="Tb_GHE")
plt.plot(np.repeat(result_dict["Borefield"]["Tb"],730), label="Tb_opt")
plt.legend(), plt.grid()
plt.subplot(nb_plots, 1, 2)
plt.plot(Tf_GHE_monthly, label="Tf_GHE")
plt.plot(np.repeat(result_dict["Borefield"]["Tf_peak_extraction"],730), label="Tf_opt")
plt.legend(), plt.grid()


In [ ]:
import pandas as pd
# Export results to an Excel file
output_excel_path = "optimization_results.xlsx"
df_results = pd.DataFrame(table, columns=["KPI", "Value", "Unit"])



# Create a Pandas Excel writer using XlsxWriter as the engine.
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    df_results.to_excel(writer, sheet_name='Results', index=False)

    # Get the xlsxwriter workbook and worksheet objects.
    workbook  = writer.book
    worksheet = writer.sheets['Results']

    # Add some cell formats.
    format1 = workbook.add_format({'num_format': '#,##0.00'})
    format2 = workbook.add_format({'bold': True, 'font_color': 'blue'})

    # Set the column width and format.
    worksheet.set_column('A:A', 20, format2)
    worksheet.set_column('B:B', 15, format1)
    worksheet.set_column('C:C', 15)

    # Add a header format.
    header_format = workbook.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'fg_color': '#D7E4BC',
        'border': 1})

    # Write the column headers with the defined format.
    for col_num, value in enumerate(df_results.columns.values):
        worksheet.write(0, col_num, value, header_format)
        

    for dev in result_dict["installed_devices"]:
        # Convert dictionary to DataFrame
        df_device = pd.DataFrame(index=range(8760))
        for key in result_dict[dev]:
            if isinstance(result_dict[dev][key], (list, np.ndarray)):
                if len(result_dict[dev][key]) == 8760:
                    array = result_dict[dev][key]
                else:
                    array = np.full(8760, np.nan)
                    array[:len(result_dict[dev][key])] = result_dict[dev][key]
                df_device[key] = array
            else: # Convert scalar to list
                df_device[key] =  np.ones(8760) * result_dict[dev][key]
        print("df_device", df_device)
        # Reset index to avoid duplicates before applying 'explode'
        df_device.reset_index(drop=True, inplace=True)

        # Check each column and explode if it's a list/array
        for column in df_device.columns:
            if isinstance(df_device[column].iloc[0], (list, np.ndarray)):
                # Apply explode and align other columns by resetting index
                df_device[column] = df_device[column].explode().reset_index(drop=True)

        # Now, the dataframe is "exploded", write it to Excel
        df_device.to_excel(writer, sheet_name=dev, index=False)
        
        # Get the worksheet object for the current device
        worksheet_device = writer.sheets[dev]
        
        # Set the column width and format for the device sheet
        for col_num, col_name in enumerate(df_device.columns):
            max_len = max(df_device[col_name].astype(str).map(len).max(), len(col_name)) + 2
            worksheet_device.set_column(col_num, col_num, max_len, format1 if col_num > 0 else format2)
        
        # Write the column headers with the defined format for the device sheet
        for col_num, value in enumerate(df_device.columns.values):
            worksheet_device.write(0, col_num, value, header_format)




In [ ]:
print(result_dict["Borefield"]["heat"])